# NEXT Transformer — Standardized EnergyBench Training

This is the canonical collaborator training notebook for the NEXT Transformer benchmark. It uses the shared EnergyBench split, training loop, checkpoint selection, evaluation metrics, and plots. The Transformer-specific choices are tokenization, positional encoding, and model architecture.

Events are streamed from the raw HDF5 files and tokenized while loading. This notebook does not require or build a token cache. Completed runs are protected from overwriting and summarized after every model.

## 1. Paths and environment

Before starting Jupyter, set `SIMPLE_ENERGYBENCH_DATA` to the extracted NEXT dataset. Optional environment variables are documented in the repository README.

In [1]:
from pathlib import Path
import json
import os
import shutil
import sys
import time

import pandas as pd
import torch
from IPython.display import display

os.environ["SIMPLE_ENERGYBENCH_DATA"] = ("/home/klz/Data/zeronu_benchmark/NEXT")
os.environ["NEXT_RUN_MODEL_IDS"] = (
    "transformer_005_summary_features_coordinate_mlp,"
    "transformer_006_summary_features_fourier_xyz"
)
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "evalutaions_workflow" / "simple_energybench").is_dir() and (
            candidate / "next_detector" / "next_transformer"
        ).is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the workflow root. Start Jupyter inside the cloned "
        "next_transformer_partner_workflow repository or set "
        "NEXT_TRANSFORMER_PROJECT_ROOT."
    )


def env_flag(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    normalized = value.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(f"{name} must be true/false or 1/0, not {value!r}")


project_setting = os.environ.get("NEXT_TRANSFORMER_PROJECT_ROOT")
PROJECT_ROOT = (
    Path(project_setting).expanduser().resolve()
    if project_setting
    else find_project_root()
)
WORKFLOW_ROOT = PROJECT_ROOT / "evalutaions_workflow"
NEXT_ROOT = PROJECT_ROOT / "next_detector"

for import_root in (WORKFLOW_ROOT, NEXT_ROOT):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

data_setting = os.environ.get("SIMPLE_ENERGYBENCH_DATA")
if data_setting is None:
    DATA_ROOT = (PROJECT_ROOT / "data" / "NEXT").resolve()
else:
    DATA_ROOT = Path(data_setting).expanduser().resolve()

OUTPUT_ROOT = Path(
    os.environ.get("NEXT_OUTPUT_ROOT", NEXT_ROOT / "results")
).expanduser().resolve()
FINAL_OUTPUT_ROOT = OUTPUT_ROOT / "final"
REFERENCE_MANIFEST_PATH = NEXT_ROOT / "results" / "event_split.json"
MANIFEST_PATH = OUTPUT_ROOT / "event_split.json"
SUMMARY_PATH = FINAL_OUTPUT_ROOT / "transformer_results.csv"

OFFICIAL_RUN = env_flag("NEXT_OFFICIAL_RUN", True)
REQUIRE_CUDA = env_flag("NEXT_REQUIRE_CUDA", OFFICIAL_RUN)
NUM_WORKERS = int(os.environ.get("NEXT_NUM_WORKERS", "8"))

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"NEXT dataset not found: {DATA_ROOT}\n"
        "Set SIMPLE_ENERGYBENCH_DATA before starting Jupyter."
    )
if not REFERENCE_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Reference split is missing: {REFERENCE_MANIFEST_PATH}")
if NUM_WORKERS < 0:
    raise ValueError("NEXT_NUM_WORKERS must be non-negative")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FINAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not MANIFEST_PATH.exists():
    shutil.copy2(REFERENCE_MANIFEST_PATH, MANIFEST_PATH)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_ROOT)
print("Outputs:", FINAL_OUTPUT_ROOT)
print("Runtime manifest:", MANIFEST_PATH)
print("Official run:", OFFICIAL_RUN)
print("Require CUDA:", REQUIRE_CUDA)
print("DataLoader workers:", NUM_WORKERS)

Project root: /home/klz/Data/zeronu_benchmark/Transformer_Approach
Dataset: /home/klz/Data/zeronu_benchmark/NEXT
Outputs: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/final
Runtime manifest: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/event_split.json
Official run: True
Require CUDA: True
DataLoader workers: 8


## 2. Shared workflow and experiment definitions

The six experiments form a 3-tokenization × 2-positional-encoding comparison. Set `NEXT_RUN_MODEL_IDS` to a comma-separated subset when distributing models across machines. The `completed_official_run` field is a manual local override: set it to `True` only for a model whose official result is already safely stored elsewhere.

In [3]:
from simple_energybench import (
    EvaluationConfig,
    TrainingConfig,
    evaluate_classification,
    prepare_dataset,
    set_seed,
    train_model,
)
from next_transformer import (
    NEXTTokenBuilder,
    NEXTTransformerClassifier,
    TokenizationConfig,
)
import next_transformer
import simple_energybench

ALL_EXPERIMENTS = [
    {
        "model_id": "transformer_001_sampled_hits_coordinate_mlp",
        "tokenization": "sampled_hits",
        "position_encoding": "coordinate_mlp",
        "completed_official_run": True,
    },
    {
        "model_id": "transformer_002_voxel_coordinate_mlp",
        "tokenization": "voxel",
        "position_encoding": "coordinate_mlp",
        "completed_official_run": True,
    },
    {
        "model_id": "transformer_003_voxel_fourier_xyz",
        "tokenization": "voxel",
        "position_encoding": "fourier_xyz",
        "completed_official_run": True,
    },
    {
        "model_id": "transformer_004_sampled_hits_fourier_xyz",
        "tokenization": "sampled_hits",
        "position_encoding": "fourier_xyz",
        "completed_official_run": True,
    },
    {
        "model_id": "transformer_005_summary_features_coordinate_mlp",
        "tokenization": "summary_features",
        "position_encoding": "coordinate_mlp",
        "completed_official_run": False,
    },
    {
        "model_id": "transformer_006_summary_features_fourier_xyz",
        "tokenization": "summary_features",
        "position_encoding": "fourier_xyz",
        "completed_official_run": False,
    },
]

requested = os.environ.get("NEXT_RUN_MODEL_IDS", "").strip()
if requested:
    run_model_ids = {item.strip() for item in requested.split(",") if item.strip()}
else:
    run_model_ids = {item["model_id"] for item in ALL_EXPERIMENTS}

known_model_ids = {item["model_id"] for item in ALL_EXPERIMENTS}
unknown_model_ids = run_model_ids - known_model_ids
if unknown_model_ids:
    raise ValueError(f"Unknown NEXT_RUN_MODEL_IDS: {sorted(unknown_model_ids)}")
selected_experiments = [
    item for item in ALL_EXPERIMENTS if item["model_id"] in run_model_ids
]
manually_completed_model_ids = {
    item["model_id"]
    for item in selected_experiments
    if item["completed_official_run"]
}
EXPERIMENTS = [
    item for item in selected_experiments if not item["completed_official_run"]
]
if not EXPERIMENTS:
    raise ValueError(
        "Every selected experiment is marked as completed. "
        "Set completed_official_run=False for at least one model."
    )

training_config = TrainingConfig(num_workers=NUM_WORKERS)
evaluation_config = EvaluationConfig()
set_seed(training_config.seed, training_config.deterministic)

tokenization_configs = {
    "voxel": TokenizationConfig(
        tokenization="voxel",
        max_tokens=512,
        voxel_size=15.0,
        coordinate_scale=1000.0,
        center_coordinates=True,
        voxel_truncation="occupancy",
        seed=training_config.seed,
    ),
    "sampled_hits": TokenizationConfig(
        tokenization="sampled_hits",
        max_tokens=512,
        voxel_size=15.0,
        coordinate_scale=1000.0,
        center_coordinates=True,
        voxel_truncation="occupancy",
        seed=training_config.seed,
    ),
    "summary_features": TokenizationConfig(
        tokenization="summary_features",
        max_tokens=512,
        voxel_size=15.0,
        coordinate_scale=1000.0,
        center_coordinates=True,
        voxel_truncation="occupancy",
        summary_morton_bits=10,
        seed=training_config.seed,
    ),
}

token_builders = {
    name: NEXTTokenBuilder(config)
    for name, config in tokenization_configs.items()
}
FEATURE_DIMS = {
    name: builder.feature_dim
    for name, builder in token_builders.items()
}
MODEL_CONFIG = {
    "d_model": 64,
    "nhead": 4,
    "num_layers": 2,
    "dim_feedforward": 256,
    "dropout": 0.1,
    "num_frequencies": 6,
}

print("EnergyBench:", simple_energybench.__file__)
print("Transformer:", next_transformer.__file__)
print("Manually marked completed:", sorted(manually_completed_model_ids))
print("Selected experiments:")
for experiment in EXPERIMENTS:
    print(" -", experiment["model_id"])
print(training_config)
print(evaluation_config)

EnergyBench: /home/klz/Data/zeronu_benchmark/Transformer_Approach/evalutaions_workflow/simple_energybench/__init__.py
Transformer: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/next_transformer/__init__.py
Manually marked completed: []
Selected experiments:
 - transformer_005_summary_features_coordinate_mlp
 - transformer_006_summary_features_fourier_xyz
TrainingConfig(batch_size=64, epochs=50, learning_rate=0.0005, weight_decay=0.0001, gradient_clip_norm=1.0, early_stopping_patience=5, early_stopping_min_delta=0.0, seed=42, deterministic=False, use_amp=True, amp_precision='auto', optimizer='adamw', scheduler='cosine', classification_loss='bce_with_logits', regression_loss='mse', device='auto', num_workers=8)
EvaluationConfig(energy_bin_width_kev=5.0, energy_grid_min_kev=0.0, energy_grid_max_kev=3000.0, energy_grid_bin_count=600, energy_unit='MeV', matching_target='overlap', min_per_class=20, min_valid_bins=2, support_trim_quantile=0.005, energy_roi=None, min_cover

## 3. Prepare the raw event streams and verify the standard split

`prepare_dataset` reads complete physics events from HDF5 and applies `NEXTTokenBuilder` while loading. All three tokenizations reuse the same event allocation. In official mode, the notebook verifies the complete event counts and exact reference split before training.

In [4]:
EXPECTED_COUNTS = {
    "total": 1_165_489,
    "train": 932_391,
    "validation": 116_549,
    "test": 116_549,
}


def split_contract(path: Path) -> dict:
    payload = json.loads(path.read_text(encoding="utf-8"))
    return {
        "settings": payload["settings"],
        "counts": payload["counts"],
        "splits": payload["splits"],
    }


data_by_tokenization = {}


def get_prepared_data(tokenization_name: str):
    if tokenization_name in data_by_tokenization:
        return data_by_tokenization[tokenization_name]

    print("Preparing raw event stream:", tokenization_name)
    prepared = prepare_dataset(
        DATA_ROOT,
        batch_size=training_config.batch_size,
        mode="classification",
        split_fractions=(0.8, 0.1, 0.1),
        seed=training_config.seed,
        num_workers=training_config.num_workers,
        manifest_path=MANIFEST_PATH,
        max_files_per_class=None,
        input_builder=token_builders[tokenization_name],
    )

    for name, expected in EXPECTED_COUNTS.items():
        if OFFICIAL_RUN and prepared.counts[name] != expected:
            raise ValueError(
                f"Official {name} count mismatch: "
                f"{prepared.counts[name]} != {expected}"
            )
    if OFFICIAL_RUN and split_contract(MANIFEST_PATH) != split_contract(
        REFERENCE_MANIFEST_PATH
    ):
        raise ValueError(
            "The runtime event allocation differs from the shared reference split. "
            "Check that everyone is using the same complete NEXT inventory."
        )

    data_by_tokenization[tokenization_name] = prepared
    return prepared


if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for this run but PyTorch cannot access it. "
        "Set NEXT_REQUIRE_CUDA=0 only for a deliberate CPU smoke test."
    )

first_data = get_prepared_data(EXPERIMENTS[0]["tokenization"])
print("Counts:", first_data.counts)
print("Manifest:", first_data.manifest_path)
print("Device request:", training_config.device)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Preflight passed.")

Preparing raw event stream: summary_features
Using cached event split manifest: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/event_split.json
Total events: 1,165,489
Train: 932,391 events (80.00%)
Validation: 116,549 events (10.00%)
Test: 116,549 events (10.00%)
Per class:
  0nubb: 648,793 total (train 519,034, validation 64,880, test 64,879)
  Bi214: 516,696 total (train 413,357, validation 51,669, test 51,670)
Boundary files shared by adjacent splits: 4
  0nubb_part_4/ATPC_0nubb_5bar_Efilt_5.0percent_smear_3983.h5
  0nubb_part_5/ATPC_0nubb_5bar_Efilt_5.0percent_smear_5204.h5
  Bi_part_6/ATPC_Bi_ion_5bar_Efilt_5.0percent_smear_3505.h5
  Bi_part_7/ATPC_Bi_ion_5bar_Efilt_5.0percent_smear_3894.h5
Counts: {'by_class': {'0nubb': {'test': 64879, 'total': 648793, 'train': 519034, 'validation': 64880}, 'Bi214': {'test': 51670, 'total': 516696, 'train': 413357, 'validation': 51669}}, 'fractions': {'test': 0.10000008580089559, 'train': 0.7999998283982088, 'validati

## 4. Train, restore the best checkpoint, and evaluate

EnergyBench selects checkpoints by validation AUC, restores the best weights, and then evaluates exactly once on the held-out test set. A completed CSV row causes that model to be skipped on restart. A partial output directory is never silently overwritten.

In [5]:
def save_summary(rows: list[dict]) -> pd.DataFrame:
    dataframe = pd.DataFrame(rows)
    temporary_path = SUMMARY_PATH.with_suffix(".csv.tmp")
    dataframe.to_csv(temporary_path, index=False)
    temporary_path.replace(SUMMARY_PATH)
    return dataframe


if SUMMARY_PATH.is_file():
    existing_results = pd.read_csv(SUMMARY_PATH)
    if existing_results["model_id"].duplicated().any():
        raise ValueError(f"Duplicate model rows in {SUMMARY_PATH}")
    experiment_rows = existing_results.to_dict(orient="records")
    completed_model_ids = set(existing_results["model_id"].astype(str))
else:
    experiment_rows = []
    completed_model_ids = set()

print("Previously completed:", sorted(completed_model_ids))

for experiment in EXPERIMENTS:
    model_id = experiment["model_id"]
    if model_id in completed_model_ids:
        print("Skipping completed model:", model_id)
        continue

    tokenization_name = experiment["tokenization"]
    position_encoding = experiment["position_encoding"]
    feature_dim = FEATURE_DIMS[tokenization_name]
    prepared_data = get_prepared_data(tokenization_name)
    run_root = FINAL_OUTPUT_ROOT / model_id

    if run_root.exists() and any(run_root.iterdir()):
        raise FileExistsError(
            f"Partial output exists for {model_id}: {run_root}. "
            "Archive or remove that one partial directory before restarting."
        )

    print("\n" + "=" * 80)
    print(model_id)
    print("Tokenization:", tokenization_name)
    print("Position encoding:", position_encoding)
    print("Content features per token:", feature_dim)
    print("Output:", run_root)
    print("=" * 80)

    set_seed(training_config.seed, training_config.deterministic)
    model = NEXTTransformerClassifier(
        position_encoding=position_encoding,
        feature_dim=feature_dim,
        **MODEL_CONFIG,
    )
    parameter_count = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    print("Trainable parameters:", f"{parameter_count:,}")

    run_root.mkdir(parents=True, exist_ok=False)
    representation_record = {
        **tokenization_configs[tokenization_name].to_dict(),
        "position_encoding": position_encoding,
        "feature_dim": feature_dim,
        **MODEL_CONFIG,
        "parameter_count": parameter_count,
        "manifest_path": str(prepared_data.manifest_path),
        "raw_hdf5_streaming": True,
        "training_config": training_config.to_dict(),
        "evaluation_config": evaluation_config.to_dict(),
    }
    (run_root / "representation_config.json").write_text(
        json.dumps(representation_record, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    training_start = time.perf_counter()
    history = train_model(
        model,
        prepared_data.train_loader,
        prepared_data.validation_loader,
        config=training_config,
        task="classification",
        output_dir=run_root / "training",
        overwrite=False,
    )
    training_seconds = time.perf_counter() - training_start

    evaluation_start = time.perf_counter()
    metrics = evaluate_classification(
        model,
        prepared_data.test_loader,
        device=training_config.device,
        output_dir=run_root / "evaluation",
        config=evaluation_config,
        overwrite=False,
    )
    evaluation_seconds = time.perf_counter() - evaluation_start

    row = {
        "model_id": model_id,
        "tokenization": tokenization_name,
        "position_encoding": position_encoding,
        "feature_dim": feature_dim,
        "parameter_count": parameter_count,
        "best_epoch": history["best_epoch"],
        "best_validation_auc": history["best_metric"],
        "test_events": metrics["n_events"],
        "inclusive_auc": metrics["auc"],
        "energy_matched_auc": metrics["matched_auc"],
        "matched_auc_status": metrics["matched_auc_status"],
        "common_support_auc": metrics["common_support_auc"],
        "shortcut_gap": metrics["shortcut_gap"],
        "energy_independence_score": metrics["energy_independence_score"],
        "worst_energy_independence_score": metrics["worst_energy_independence_score"],
        "training_seconds": training_seconds,
        "evaluation_seconds": evaluation_seconds,
    }
    experiment_rows.append(row)
    results_dataframe = save_summary(experiment_rows)
    display(pd.DataFrame([row]))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Previously completed: []

transformer_005_summary_features_coordinate_mlp
Tokenization: summary_features
Position encoding: coordinate_mlp
Content features per token: 4
Output: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/final/transformer_005_summary_features_coordinate_mlp
Trainable parameters: 111,233
Epoch 001/050 | train loss 0.433555 | val loss 0.978066 | val auc 0.898287
Epoch 002/050 | train loss 0.35121 | val loss 0.272375 | val auc 0.953222
Epoch 003/050 | train loss 0.293425 | val loss 0.263946 | val auc 0.960941
Epoch 004/050 | train loss 0.268684 | val loss 0.319715 | val auc 0.960953
Epoch 005/050 | train loss 0.256073 | val loss 0.250756 | val auc 0.965648
Epoch 006/050 | train loss 0.239917 | val loss 0.218232 | val auc 0.969508
Epoch 007/050 | train loss 0.230226 | val loss 0.244707 | val auc 0.969076
Epoch 008/050 | train loss 0.222397 | val loss 0.400629 | val auc 0.969182
Epoch 009/050 | train loss 0.212923 | val loss 0.19915 | val auc 

,model_id,tokenization,position_encoding,feature_dim,parameter_count,best_epoch,best_validation_auc,test_events,inclusive_auc,energy_matched_auc,matched_auc_status,common_support_auc,shortcut_gap,energy_independence_score,worst_energy_independence_score,training_seconds,evaluation_seconds
0,transformer_005_summary_features_coordinate_mlp,summary_features,coordinate_mlp,4,111233,43,0.98799,116549,0.987607,0.987424,ok,0.987644,0.00022,0.970741,0.961919,21147.4624,23.762902



transformer_006_summary_features_fourier_xyz
Tokenization: summary_features
Position encoding: fourier_xyz
Content features per token: 4
Output: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/final/transformer_006_summary_features_fourier_xyz
Trainable parameters: 113,537
Epoch 001/050 | train loss 0.376536 | val loss 0.633913 | val auc 0.915831
Epoch 002/050 | train loss 0.322785 | val loss 0.27457 | val auc 0.951075
Epoch 003/050 | train loss 0.27626 | val loss 0.254767 | val auc 0.960648
Epoch 004/050 | train loss 0.249656 | val loss 0.283911 | val auc 0.964866
Epoch 005/050 | train loss 0.236437 | val loss 0.224977 | val auc 0.96814
Epoch 006/050 | train loss 0.224746 | val loss 0.209886 | val auc 0.970781
Epoch 007/050 | train loss 0.219134 | val loss 0.209321 | val auc 0.972897
Epoch 008/050 | train loss 0.209848 | val loss 0.262403 | val auc 0.971726
Epoch 009/050 | train loss 0.204839 | val loss 0.201496 | val auc 0.973566
Epoch 010/050 | train loss

,model_id,tokenization,position_encoding,feature_dim,parameter_count,best_epoch,best_validation_auc,test_events,inclusive_auc,energy_matched_auc,matched_auc_status,common_support_auc,shortcut_gap,energy_independence_score,worst_energy_independence_score,training_seconds,evaluation_seconds
0,transformer_006_summary_features_fourier_xyz,summary_features,fourier_xyz,4,113537,42,0.984159,116549,0.983707,0.983448,ok,0.983679,0.000232,0.972612,0.965932,21116.24895,23.827362


## 5. Completed-run summary

Use `next_energybench_results.ipynb` for the complete artifact audit and presentation figures.

In [6]:
if SUMMARY_PATH.is_file():
    final_results = pd.read_csv(SUMMARY_PATH).sort_values(
        "energy_matched_auc", ascending=False
    ).reset_index(drop=True)
    display(final_results)
    print("Saved summary:", SUMMARY_PATH)
else:
    print("No model has completed training and evaluation yet.")

,model_id,tokenization,position_encoding,feature_dim,parameter_count,best_epoch,best_validation_auc,test_events,inclusive_auc,energy_matched_auc,matched_auc_status,common_support_auc,shortcut_gap,energy_independence_score,worst_energy_independence_score,training_seconds,evaluation_seconds
0,transformer_005_summary_features_coordinate_mlp,summary_features,coordinate_mlp,4,111233,43,0.987990,116549,0.987607,0.987424,ok,0.987644,0.000220,0.970741,0.961919,21147.46240,23.762902
1,transformer_006_summary_features_fourier_xyz,summary_features,fourier_xyz,4,113537,42,0.984159,116549,0.983707,0.983448,ok,0.983679,0.000232,0.972612,0.965932,21116.24895,23.827362


Saved summary: /home/klz/Data/zeronu_benchmark/Transformer_Approach/next_detector/results/final/transformer_results.csv
